In [3]:
#!/usr/bin/env python3

#SBATCH --account=pi-hcn1
#SBATCH --time=03:30:00 # 2 hrs enough for almost all
#SBATCH --partition=bigmem2
#SBATCH --ntasks=1
#SBATCH --mem-per-cpu=300G # 300G enough for most
#SBATCH --mail-type=all
#SBATCH --mail-user=letitiayhho@rcc.uchicago.edu
#SBATCH --output=logs/preprocess_erp-%j.log

import numpy as np
import os.path as op
import os
import sys

cwd = os.getcwd()
sys.path.append(cwd)
print(sys.path)

from pprint import pformat
import argparse
# EEG utilities
import mne
from mne.preprocessing import ICA, create_eog_epochs
from pyprep.prep_pipeline import PrepPipeline
from autoreject import get_rejection_threshold, validation_curve
# BIDS utilities
from mne_bids import BIDSPath, read_raw_bids
from util.io.bids import DataSink

# constants
BIDS_ROOT = '../data/bids'
DERIV_ROOT = op.join(BIDS_ROOT, 'derivatives')
ERP_PASSBAND = (0.1, 40)
TASK = 'pitch'
TMIN = -0.2
TMAX = 0.5

sub = '43'
run = '1'
'''
Parameters
----------
sub : str
    Subject ID as in BIDS dataset
'''
print('----------------- load data ------------------')
bids_path = BIDSPath(
    root = BIDS_ROOT,
    subject = sub,
    task = TASK,
    run = run,
    datatype = 'eeg'
    )
print(bids_path)
raw = read_raw_bids(bids_path, verbose = False)
events, event_ids = mne.events_from_annotations(raw)

print('----------------- re-reference eye electrodes to become bipolar EOG ------------------')
raw.load_data()
def reref(dat):
    dat[0,:] = (dat[1,:] - dat[0,:])
    return dat
raw = raw.apply_function(
    reref,
    picks = ['leog', 'Fp2'],
    channel_wise = False
)
raw = raw.apply_function(
    reref,
    picks = ['reog', 'Fp1'],
    channel_wise = False
)
raw = raw.set_channel_types({'leog': 'eog', 'reog': 'eog'})

# print('----------------- run PREP pipeline ------------------') # notch, exclude bad chans, and re-reference
# raw.load_data()
# np.random.seed(int(sub))
# lf = raw.info['line_freq']
# prep_params = {
#     "ref_chs": "eeg",
#     "reref_chs": "eeg",
#     "line_freqs": np.arange(lf, ERP_PASSBAND[1], lf)
# }
# prep = PrepPipeline(
#     raw,
#     prep_params,
#     raw.get_montage(),
#     ransac = False,
#     random_state = int(sub)
#     )
# prep.fit()

# print('----------------- Extract data from PREP ------------------')
# prep_eeg = prep.raw_eeg # get EEG channels from PREP
# prep_non_eeg = prep.raw_non_eeg # get non-EEG channels from PREP
# raw_data = np.concatenate((prep_eeg.get_data(), prep_non_eeg.get_data())) # combine data from the two

# # Create info object for post-PREP data
# print('Create info object for post-PREP data')
# new_ch_names = prep_eeg.info['ch_names'] + prep_non_eeg.info['ch_names']
# raw = raw.reorder_channels(new_ch_names) # modify the channel names on the original raw data
# raw_info = raw.info # use the modified info from the original raw data object
 
# # Combine post-prep data and new info
# print('Create new raw object')
# raw = mne.io.RawArray(raw_data, raw_info) # replace original raw object

# print('----------------- Filter ------------------') 
# raw = raw.filter(*ERP_PASSBAND)

# ## now prepare non-epoched data for ERP analysis
# # identify bad ICs on weakly highpassed data
# print('----------------- Epoch data for ERP analysis ------------------')
# epochs = mne.Epochs(
#     raw,
#     events, # same events as FFR epochs
#     tmin = TMIN,
#     tmax = TMAX, # only prestim
#     event_id = event_ids,
#     baseline = None,
#     preload = True
# )

# print('----------------- Downsample ------------------') 
# epochs = epochs.resample(1000) # resample after epoching to avoid adding jitter to event triggers

# print('----------------- Run ICA ------------------')
# ica = ICA(n_components = 15, random_state = 0)
# ica.fit(epochs, picks = ['eeg', 'eog'])

# print('----------------- Apply ICA ------------------')
# eog_indices, eog_scores = ica.find_bads_eog(epochs, threshold = 1.96)
# ica.exclude = eog_indices
# ica.apply(epochs) # transforms in place 

# if ica.exclude: # if we found any bad components
#     fig_ica_removed = ica.plot_components(ica.exclude)

# # now we no longer need EOG channels
# epochs = epochs.drop_channels('leog')
# epochs = epochs.drop_channels('reog')

# # Keep only midline channels
# # epochs = epochs.pick_channels(ch_names = ['Fz', 'FCz', 'Cz', 'CPz', 'Pz'])

# print('----------------- Baseline correct ------------------')
# epochs = epochs.apply_baseline((TMIN, 0.))

# print('----------------- Reject bad trials ------------------')
# thres = get_rejection_threshold(epochs)
# print(thres)
# epochs.drop_bad(reject = thres)

# print('----------------- Save ------------------')
# sink = DataSink(DERIV_ROOT, 'erp')
# erp_fpath = sink.get_path(
#     subject = sub,
#     task = TASK,
#     run = run,
#     desc = 'forERPall',
#     suffix = 'epo',
#     extension = 'fif.gz'
# )
# print(f'Saving epochs for ERP analysis to: {erp_fpath}')
# epochs.save(erp_fpath, overwrite = True)

# print('----------------- generate a report ------------------')
# report = mne.Report(verbose = True)
# report.parse_folder(op.dirname(erp_fpath), pattern = '*epo.fif.gz', render_bem = False)
# if ica.exclude:
#     fig_ica_removed = ica.plot_components(ica.exclude, show = False)
#     report.add_figure(
#         fig_ica_removed,
#         title = 'Removed ICA Components',
#         section = 'ICA'
#     )
# bads = prep.noisy_channels_original
# html_lines = []
# for line in pformat(bads).splitlines():
#     html_lines.append('<br/>%s' % line)
# html = '\n'.join(html_lines)
# report.add_html(html, title = 'Interpolated Channels', section = 'channels')
# report.add_html(epochs.info._repr_html_(), title = 'Epochs Info (FFR)', section = 'info')
# report.add_html(epochs.info._repr_html_(), title = 'Epochs Info (ERP)', section = 'info')
# report.save(op.join(sink.deriv_root, 'sub-%s-all.html'%sub), overwrite = True)


['/project/hcn1/.conda/envs/mne/lib/python311.zip', '/project/hcn1/.conda/envs/mne/lib/python3.11', '/project/hcn1/.conda/envs/mne/lib/python3.11/lib-dynload', '', '/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages', '/project/hcn1/Letty/pitch_tracking_attention/analysis']
----------------- load data ------------------
../data/bids/sub-43/eeg/sub-43_task-pitch_run-1_eeg.vhdr
Used Annotations descriptions: ['11', '12', '13', '21', '22', '23', '31', '32', '33']
----------------- re-reference eye electrodes to become bipolar EOG ------------------
Reading 0 ... 20290249  =      0.000 ...  4058.050 secs...


/scratch/local/jobs/43583663/ipykernel_741280/2105721134.py:57: RuntimeWarning: The unit for channel(s) Aux1 has changed from NA to V.
  raw = read_raw_bids(bids_path, verbose = False)
/scratch/local/jobs/43583663/ipykernel_741280/2105721134.py:57: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['leog', 'reog', 'Aux1']
  raw = read_raw_bids(bids_path, verbose = False)
/scratch/local/jobs/43583663/ipykernel_741280/2105721134.py:57: RuntimeWarning: Not setting position of 1 stim channel found in montage:
['Aux1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = read_raw_bids(bids_path, verbose = False)


In [ ]:
# print('----------------- run PREP pipeline ------------------') # notch, exclude bad chans, and re-reference
# raw.load_data()
# np.random.seed(int(sub))
# lf = raw.info['line_freq']
# prep_params = {
#     "ref_chs": "eeg",
#     "reref_chs": "eeg",
#     "line_freqs": np.arange(lf, ERP_PASSBAND[1], lf)
# }
prep = PrepPipeline(
    raw,
    prep_params,
    raw.get_montage(),
    ransac = False,
    random_state = int(sub)
    )
prep.fit()

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/project/hcn1/.conda/envs/mne/lib/python3.11/site-packages/pyprep/prep_pipeline.py:148: FutureWarning: The default for pick_channels will change from ordered=False to ordered=True in 1.5 and this will result in a change of behavior because the resulting channel order will not match. Either use a channel order that matches your instance or pass ordered=False.
  self.raw_non_eeg.pick_channels(self.ch_names_non_eeg)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 16501 samples (3.300 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:   14.4s


In [5]:
print(raw)
print(raw.get_channel_types(picks=None, unique=True))
print(len(mne.pick_types(raw.info, eeg=True)), "EEG channels")


<RawBrainVision | sub-43_task-pitch_run-1_eeg.eeg, 65 x 20290250 (4058.0 s), ~9.83 GB, data loaded>
['eeg', 'eog', 'stim']
62 EEG channels


In [4]:
raw.info["bads"]

[]

In [1]:
import os
import sys
import subprocess
import argparse
from bids import BIDSLayout
from util.io.iter_BIDSPaths import *
from util.io.bids import DataSink

import gc
import sys
import mne
import numpy as np
import pandas as pd

# import matplotlib.pyplot as plt
# from typing import Tuple, Iterator
# from mne_bids import BIDSPath, read_raw_bids, print_dir_tree
# from mne.time_frequency import tfr_morlet
# from bids import BIDSLayout

# from sklearn.pipeline import make_pipeline
# from sklearn import preprocessing
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# from mne.decoding import SlidingEstimator, cross_val_multiscore

In [21]:
BIDS_ROOT = '../data/bids'
FIGS_ROOT = '../figs'
STIM_FREQS = np.array([130, 200, 280])

cond = 'tone_target'

In [15]:
print("---------- Load data ----------")
fpath = '/project2/hcn1/pitch_tracking_attention/data/bids/derivatives/preprocessing/sub-16/sub-16_task-pitch_run-1_desc-clean_epo.fif.gz'
epochs = mne.read_epochs(fpath)
print(epochs.event_id)

---------- Load data ----------
Reading /project2/hcn1/pitch_tracking_attention/data/bids/derivatives/preprocessing/sub-16/sub-16_task-pitch_run-1_desc-clean_epo.fif.gz ...
    Found the data of interest:
        t =    -200.00 ...     350.00 ms
        0 CTF compensation matrices available
Reading /project2/hcn1/pitch_tracking_attention/data/bids/derivatives/preprocessing/sub-16/sub-16_task-pitch_run-1_desc-clean_epo.fif-1.gz ...
    Found the data of interest:
        t =    -200.00 ...     350.00 ms
        0 CTF compensation matrices available
0 bad epochs dropped
0 bad epochs dropped
Not setting metadata
3831 matching events found
No baseline correction applied
0 projection items activated
{'11': 10001, '12': 10002, '13': 10003, '21': 10004, '22': 10005, '23': 10006, '31': 10007, '32': 10008, '33': 10009}


In [22]:
print("---------- Subset epochs ----------")
if cond == 'target':
    condition_epochs = epochs
elif cond == 'tone_target':
    CONDS = ['11', '22', '33']
    condition_epochs = epochs[CONDS]
elif cond == 'tone_nontarget':
    CONDS = ['12', '13', '21', '23', '31', '32']
    condition_epochs = epochs[CONDS]
else:
    CONDS = {'1': ['11', '12', '13'], # subset the trials belonging to each target tone
             '2': ['21', '22', '23'],
             '3': ['31', '32', '33'],}
    condition_epochs = epochs[CONDS[cond[0]]]
events = condition_epochs.events
print(condition_epochs.event_id)
print(condition_epochs)

---------- Subset epochs ----------
{'11': 10001, '22': 10005, '33': 10009}
<EpochsFIF |  1265 events (all good), -0.2 - 0.35 sec, baseline -0.2 – 0 sec, ~1.61 GB, data loaded,
 '11': 377
 '22': 468
 '33': 420>


In [23]:
labels = pd.Series(events[:, 2])
EVENT_DICTS = {'tone_target': {10001 : 1, 10005: 2, 10009: 3},
               'tone_nontarget': {10002 : 1, 10003 : 1, 10004: 2, 10006: 2, 10007: 3, 10008: 3},
               '11': {10001: 1, 10002: 0, 10003: 0},
               '12': {10001: 0, 10002: 1, 10003: 0},
               '13': {10001: 0, 10002: 0, 10003: 1},
               '21': {10004: 1, 10005: 0, 10006: 0},
               '22': {10004: 0, 10005: 1, 10006: 0},
               '23': {10004: 0, 10005: 0, 10006: 1},
               '31': {10007: 1, 10008: 0, 10009: 0},
               '32': {10007: 0, 10008: 1, 10009: 0},
               '33': {10007: 0, 10008: 0, 10009: 1},
               'target': {10001 : 1, 10002 : 0, 10003 : 0, 10004: 0, 10005: 1, 10006: 0, 10007: 0, 10008: 0, 10009: 1}}
                # FOR REFERENCE {'11': 10001, '12': 10002, '13': 10003, '21': 10004, 
                #'22': 10005, '23': 10006, '31': 10007, '32': 10008, '33': 10009}
y = labels.replace(EVENT_DICTS[cond])
print(labels)
print(y)

0       10001
1       10001
2       10001
3       10001
4       10001
        ...  
1260    10005
1261    10005
1262    10005
1263    10005
1264    10005
Length: 1265, dtype: int32
0       1
1       1
2       1
3       1
4       1
       ..
1260    2
1261    2
1262    2
1263    2
1264    2
Length: 1265, dtype: int32


In [ ]:
print("---------- Compute power ----------")
n_cycles = STIM_FREQS / 7 # different number of cycle per frequency
                           # higher constant, fewer windows, maybe?
power = tfr_morlet(epochs,
                   freqs = STIM_FREQS,
                   n_cycles = n_cycles,
                   use_fft = True,
                   return_itc = False,
                   decim = 3,
                   n_jobs = 1,
                   average = False)
power = np.log10(power)

del epochs
gc.collect()

# Get some information
n_epochs = np.shape(power)[0]
n_channels = np.shape(power)[1]
n_freqs = np.shape(power)[2]
n_windows = np.shape(power)[3]
print("n_windows: " + str(n_windows))

---------- Compute power ----------
Not setting metadata


In [ ]:
print("---------- Prepare for decoder ----------")
# Reshape for classifier
X = power.reshape((n_epochs, n_freqs * n_channels, n_windows)) # Set order to preserve epoch order

# Create array of condition labels
labels = pd.Series(events[:, 2])
y = labels.replace({10001 : 130, 10002 : 200, 10003 : 280})
le = preprocessing.LabelEncoder()
y = le.fit_transform(y)

In [ ]:
print("---------- Decode ----------")
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver = 'liblinear')
)

print("Creating sliding estimators")
time_decod = SlidingEstimator(clf)

print("Fit estimators")
scores = cross_val_multiscore(
    time_decod,
    X, # a trials x features x time array
    y, # an (n_trials,) array of integer condition labels
    cv = 5, # use stratified 5-fold cross-validation
    n_jobs = -1, # use all available CPU cores
)
scores = np.mean(scores, axis = 0) # average across cv splits

In [ ]:
print("---------- Save decoder scores ----------")
print('Saving scores to: ' + scores_fpath)
np.save(scores_fpath, scores)

---------- Load data ----------
/project2/hcn1/pitch_tracking_attention/data/bids/derivatives/preprocessing/sub-12/sub-12_task-pitch_run-1_desc-clean_epo.fif.gz
Reading /project2/hcn1/pitch_tracking_attention/data/bids/derivatives/preprocessing/sub-12/sub-12_task-pitch_run-1_desc-clean_epo.fif.gz ...
    Found the data of interest:
        t =    -200.00 ...     350.00 ms
        0 CTF compensation matrices available
Reading /project2/hcn1/pitch_tracking_attention/data/bids/derivatives/preprocessing/sub-12/sub-12_task-pitch_run-1_desc-clean_epo.fif-1.gz ...
    Found the data of interest:
        t =    -200.00 ...     350.00 ms
        0 CTF compensation matrices available
0 bad epochs dropped
0 bad epochs dropped
Not setting metadata
3350 matching events found
No baseline correction applied
0 projection items activated
---------- Compute power ----------
Not setting metadata


In [ ]:
print("---------- Plot ----------")
n_stimuli = 3
fig, ax = plt.subplots()
ax.plot(range(len(scores)), scores, label = 'score')
ax.axhline(1/n_stimuli, color = 'k', linestyle = '--', label = 'chance')
ax.set_xlabel('Times')
ax.set_ylabel('Accuracy')  # Area Under the Curve
ax.legend()
ax.set_title('Sensor space decoding')

# Save plot
fig_fpath = FIGS_ROOT + '/subj-' + sub + '_' + 'task-pitch_' + 'run-' + run + '_log_reg_no_crop' + '.png'
print('Saving figure to: ' + fig_fpath)
plt.savefig(fig_fpath)